# CKD-AHSS: Adaptive Hardness- and Safety-Aware Hybrid Sampling

**Purpose:** implement and evaluate an experimental custom sampler for a severely imbalanced CKD dataset.

The notebook compares the proposed sampler with the original data, Random Oversampling, SMOTE, Borderline-SMOTE, ADASYN, SMOTEENN, SMOTETomek, Random Under-sampling, and class weighting.

> **Research integrity:** CKD-AHSS is a proposed experimental design, not a proven state-of-the-art method. This notebook does not guarantee that it will outperform existing samplers. Its value must be established using leakage-safe experiments, repeated validation, ablation studies, and ideally an independent cohort.

**Important:** The dataset has 1,659 records, 54 columns, and an approximately 91.86% CKD / 8.14% non-CKD distribution. Do not use the hold-out test set to tune this sampler.

## 1. Install dependencies

Run this cell once. On Kaggle, only `imbalanced-learn` is installed if missing; pandas, numpy and scikit-learn are left untouched.

In [ ]:
%pip -q install imbalanced-learn

## 2. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import (
    balanced_accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report, make_scorer
)
from sklearn.metrics.pairwise import manhattan_distances
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.utils import check_random_state

from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ready.")

## 3. Load your CKD dataset

The dataset is read directly from the Kaggle input path.

In [ ]:
CSV_PATH = "/kaggle/input/datasets/rabieelkharoua/chronic-kidney-disease-dataset-analysis/Chronic_Kidney_Dsease_data.csv"

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
TARGET_COLUMN = "Diagnosis"
POSITIVE_LABEL = "1"
DROP_COLUMNS = ["PatientID", "DoctorInCharge"]

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column {TARGET_COLUMN!r} not found. Available: {df.columns.tolist()}")

print(df[TARGET_COLUMN].value_counts(dropna=False))
print("Unique labels:", df[TARGET_COLUMN].dropna().unique())

## 4. Clean labels and create a stratified hold-out test set

`Diagnosis` is the target (`1` = CKD, `0` = non-CKD). `PatientID` (identifier) and `DoctorInCharge` (constant) are dropped. The sampler oversamples whichever class is less frequent (non-CKD). The hold-out test data are kept untouched: sampling, preprocessing, and model fitting happen only inside training folds.

In [ ]:
def normalize_label(v):
    s = str(v).strip().lower()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s

df = df.loc[df[TARGET_COLUMN].notna()].reset_index(drop=True)
label_norm = df[TARGET_COLUMN].map(normalize_label)
y = (label_norm == normalize_label(POSITIVE_LABEL)).astype(int)

if y.nunique() < 2:
    raise ValueError("Could not identify both classes. Check POSITIVE_LABEL.")

X = df.drop(columns=[TARGET_COLUMN] + [c for c in DROP_COLUMNS if c in df.columns]).copy()
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

print("Features:", X.shape[1])
print("CKD (1):", int(y.sum()))
print("Non-CKD (0):", int((y == 0).sum()))
print("Imbalance ratio (majority/minority):", round(y.value_counts().max() / y.value_counts().min(), 2))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Training class counts:\n", y_train.value_counts())

## 5. CKD-AHSS sampler: design

The sampler operates on the **raw mixed-type DataFrame**, before model preprocessing.

1. **Neighbourhood safety:** estimate the proportion of minority neighbours around each minority observation.
2. **Cluster-aware generation:** generate synthetic cases only between minority neighbours assigned to the same cluster.
3. **Hardness-aware selection:** prioritize minority observations whose neighbourhood contains both classes, while excluding very unsafe/noisy points.
4. **Selective majority cleaning:** remove only majority observations with strong local minority dominance.
5. **Adaptive ratio:** `sampling_strategy` controls the desired minority/majority ratio. It is selected using training-side validation, not the hold-out test set.

Numeric features are interpolated; categorical features are copied from one parent observation. Missing values are temporarily handled for distance calculations using training-fold medians/modes. The downstream model pipeline independently imputes and encodes data.

In [ ]:
def is_categorical(s):
    return (not pd.api.types.is_numeric_dtype(s)) or pd.api.types.is_bool_dtype(s)


class CKDAHSSSampler(BaseEstimator):
    def __init__(self, sampling_strategy=0.50, k_neighbors=5, n_clusters=4,
                 safety_min=0.20, hardness_focus=0.50, clean_threshold=0.75,
                 clean_majority=False, random_state=42):
        self.sampling_strategy = sampling_strategy
        self.k_neighbors = k_neighbors
        self.n_clusters = n_clusters
        self.safety_min = safety_min
        self.hardness_focus = hardness_focus
        self.clean_threshold = clean_threshold
        self.clean_majority = clean_majority
        self.random_state = random_state

    def _prepare(self, X):
        self.columns_ = list(X.columns)
        self.cat_cols_ = [c for c in X.columns if is_categorical(X[c])]
        self.num_cols_ = [c for c in X.columns if c not in self.cat_cols_]
        self.int_cols_ = []

        Z = X.copy()
        for c in self.num_cols_:
            Z[c] = pd.to_numeric(Z[c], errors="coerce").astype(float)
            vals = Z[c].dropna().to_numpy()
            if len(vals) and np.all(np.mod(vals, 1) == 0):
                self.int_cols_.append(c)
            med = Z[c].median()
            Z[c] = Z[c].fillna(0.0 if pd.isna(med) else med)
        for c in self.cat_cols_:
            col = Z[c].astype("object")
            mode = col.dropna().mode()
            fill = mode.iloc[0] if len(mode) else "__MISSING__"
            Z[c] = col.where(col.notna(), fill)
        return Z

    def _distance_matrix(self, Z):
        n = len(Z)
        parts = []
        if self.num_cols_:
            A = Z[self.num_cols_].to_numpy(dtype=float)
            mins, maxs = A.min(axis=0), A.max(axis=0)
            ranges = np.where((maxs - mins) == 0, 1.0, maxs - mins)
            A = (A - mins) / ranges
            parts.append(manhattan_distances(A) / A.shape[1])
        if self.cat_cols_:
            C = Z[self.cat_cols_].astype(str).to_numpy()
            D_cat = np.zeros((n, n), dtype=float)
            for j in range(C.shape[1]):
                D_cat += (C[:, j, None] != C[None, :, j])
            parts.append(D_cat / C.shape[1])
        D = sum(parts) / len(parts)
        np.fill_diagonal(D, np.inf)
        return D

    def _cluster_labels(self, Z, minority_idx):
        parts = []
        if self.num_cols_:
            A = Z[self.num_cols_].to_numpy(dtype=float)
            mu, sd = A.mean(axis=0), A.std(axis=0)
            sd[sd == 0] = 1
            parts.append((A - mu) / sd)
        if self.cat_cols_:
            parts.append(pd.get_dummies(Z[self.cat_cols_].astype(str)).to_numpy(dtype=float))
        if not parts or len(minority_idx) < 3:
            return np.zeros(len(Z), dtype=int)
        M = np.concatenate(parts, axis=1)
        k = max(1, min(int(self.n_clusters), len(minority_idx)))
        km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
        km.fit(M[minority_idx])
        return km.predict(M)

    def fit_resample(self, X, y):
        rng = check_random_state(self.random_state)
        X0 = (X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)).reset_index(drop=True)
        y0 = np.asarray(y).astype(int)
        if len(X0) != len(y0):
            raise ValueError("X and y have different lengths.")
        if set(np.unique(y0)) != {0, 1}:
            raise ValueError("CKDAHSSSampler requires binary labels encoded as 0 and 1.")

        counts = np.bincount(y0, minlength=2)
        minority_label = int(np.argmin(counts))
        majority_label = 1 - minority_label

        Z = self._prepare(X0)
        D = self._distance_matrix(Z)
        minority_idx = np.flatnonzero(y0 == minority_label)
        majority_idx = np.flatnonzero(y0 == majority_label)
        n_min, n_maj = len(minority_idx), len(majority_idx)
        if n_min < 2:
            raise ValueError("At least two minority observations are required.")

        k = min(max(1, int(self.k_neighbors)), len(y0) - 1)
        neighbour_idx = np.argpartition(D, k - 1, axis=1)[:, :k]
        safety = (y0[neighbour_idx] == minority_label).mean(axis=1)
        minority_safety = safety[minority_idx]

        eligible = minority_idx[minority_safety >= float(self.safety_min)]
        if len(eligible) < 2:
            eligible = minority_idx[np.argsort(minority_safety)[-2:]]
        clusters = self._cluster_labels(Z, minority_idx)

        hard = np.clip(1.0 - np.abs(minority_safety - 0.5) / 0.5, 0, 1)
        safety_weight = np.clip(minority_safety, 0.05, 1.0)
        weights_all = (0.25 + float(self.hardness_focus) * hard) * safety_weight
        weights = weights_all[np.searchsorted(minority_idx, eligible)].astype(float)
        if not np.isfinite(weights).all() or weights.sum() <= 0:
            weights = np.ones(len(eligible), dtype=float)
        weights = weights / weights.sum()

        n_generate = max(0, int(np.ceil(float(self.sampling_strategy) * n_maj)) - n_min)

        synth = None
        if n_generate > 0:
            kn = max(1, int(self.k_neighbors))
            candidates = {}
            for a in eligible:
                pool = eligible[(eligible != a) & (clusters[eligible] == clusters[a])]
                if len(pool) == 0:
                    pool = eligible[eligible != a]
                if len(pool) == 0:
                    pool = minority_idx[minority_idx != a]
                candidates[a] = pool[np.argsort(D[a, pool])[:kn]]

            anchors = rng.choice(eligible, size=n_generate, p=weights)
            partners = np.array([rng.choice(candidates[a]) for a in anchors])
            lam = rng.uniform(0.15, 0.85, size=n_generate)

            pieces = []
            if self.num_cols_:
                A = Z[self.num_cols_].to_numpy(dtype=float)
                S = A[anchors] + lam[:, None] * (A[partners] - A[anchors])
                synth_num = pd.DataFrame(S, columns=self.num_cols_)
                for c in self.int_cols_:
                    synth_num[c] = synth_num[c].round()
                pieces.append(synth_num)
            if self.cat_cols_:
                Cm = Z[self.cat_cols_].to_numpy(dtype=object)
                pick = rng.rand(n_generate, Cm.shape[1]) < 0.5
                pieces.append(pd.DataFrame(np.where(pick, Cm[anchors], Cm[partners]),
                                           columns=self.cat_cols_))
            synth = pd.concat(pieces, axis=1)[self.columns_]

        keep = np.ones(len(y0), dtype=bool)
        if self.clean_majority and n_maj:
            mixed = (y0[neighbour_idx[majority_idx]] == minority_label).mean(axis=1)
            keep[majority_idx[mixed >= float(self.clean_threshold)]] = False

        X_kept = X0.loc[keep].reset_index(drop=True)
        if synth is not None:
            X_res = pd.concat([X_kept, synth], ignore_index=True)
        else:
            X_res = X_kept
        y_res = np.concatenate([y0[keep], np.full(n_generate, minority_label, dtype=int)])

        self.safety_scores_ = safety
        self.cluster_labels_ = clusters
        self.n_generated_ = int(n_generate)
        self.n_removed_majority_ = int((~keep).sum())
        return X_res, y_res

## 6. Define preprocessing and classifiers

Imputation and encoding are fitted inside each training fold. The sampler runs before preprocessing and sees only that fold's training data. The test fold is never resampled.

In [ ]:
def make_preprocessor(X_sample):
    cat_cols = [c for c in X_sample.columns if is_categorical(X_sample[c])]
    num_cols = [c for c in X_sample.columns if c not in cat_cols]

    num_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ], remainder="drop")

def build_model(model_name="HistGradientBoosting", class_weight=False):
    cw = "balanced" if class_weight else None
    if model_name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight=cw,
            random_state=RANDOM_STATE, n_jobs=-1
        )
    if model_name == "ExtraTrees":
        return ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight=cw,
            random_state=RANDOM_STATE, n_jobs=-1
        )
    if model_name == "LogisticRegression":
        return LogisticRegression(max_iter=3000, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "SVM":
        return SVC(probability=True, class_weight=cw, random_state=RANDOM_STATE)
    raise ValueError("Unknown model")

def make_pipeline(sampler, model_name="HistGradientBoosting", class_weight=False, X_reference=None):
    if X_reference is None:
        X_reference = X_train
    return ImbPipeline([
        ("sampler", sampler),
        ("preprocess", make_preprocessor(X_reference)),
        ("model", build_model(model_name, class_weight=class_weight))
    ])

## 7. Quick sampler sanity check

This is a diagnostic on the training partition only. It is not a performance result.

In [ ]:
ahss = CKDAHSSSampler(
    sampling_strategy=0.50, k_neighbors=5, n_clusters=4,
    safety_min=0.20, hardness_focus=0.50,
    clean_majority=False, random_state=RANDOM_STATE
)
X_ahss, y_ahss = ahss.fit_resample(X_train, y_train)
print("Before:", pd.Series(y_train).value_counts().to_dict())
print("After: ", pd.Series(y_ahss).value_counts().to_dict())
print("Synthetic rows generated:", ahss.n_generated_)
print("Majority rows removed:", ahss.n_removed_majority_)
print("Sampler output shape:", X_ahss.shape)

## 8. Compare samplers using repeated stratified cross-validation

Use the same folds and classifier for each sampler. The example uses 5 folds × 3 repeats. For a final paper, consider more repeats and nested CV if computational resources permit.

**Threshold handling:** this first comparison uses the default 0.5 probability threshold. A separate training-side threshold optimization is provided later. Do not choose thresholds on the hold-out test set.

In [ ]:
sampler_factories = {
    "No sampling": lambda: "passthrough",
    "Class weighting": lambda: "passthrough",
    "Random oversampling": lambda: RandomOverSampler(random_state=RANDOM_STATE),
    "SMOTE": lambda: SMOTE(k_neighbors=5, random_state=RANDOM_STATE),
    "Borderline-SMOTE": lambda: BorderlineSMOTE(k_neighbors=5, random_state=RANDOM_STATE),
    "ADASYN": lambda: ADASYN(n_neighbors=5, random_state=RANDOM_STATE),
    "SMOTEENN": lambda: SMOTEENN(random_state=RANDOM_STATE),
    "SMOTETomek": lambda: SMOTETomek(random_state=RANDOM_STATE),
    "Random undersampling": lambda: RandomUnderSampler(random_state=RANDOM_STATE),
    "CKD-AHSS": lambda: CKDAHSSSampler(
        sampling_strategy=0.50, k_neighbors=5, n_clusters=4,
        safety_min=0.20, hardness_focus=0.50,
        clean_majority=False, random_state=RANDOM_STATE
    )
}

scoring = {
    "bal_acc": make_scorer(balanced_accuracy_score),
    "recall_ckd": make_scorer(recall_score, pos_label=1, zero_division=0),
    "precision_ckd": make_scorer(precision_score, pos_label=1, zero_division=0),
    "f1_ckd": make_scorer(f1_score, pos_label=1, zero_division=0),
    "mcc": make_scorer(matthews_corrcoef)
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
MODEL_NAME = "HistGradientBoosting"

cv_results = []
for sampler_name, factory in sampler_factories.items():
    print("Evaluating:", sampler_name)
    pipe = make_pipeline(
        factory(), MODEL_NAME,
        class_weight=(sampler_name == "Class weighting"),
        X_reference=X_train
    )
    try:
        scores = cross_validate(
            pipe, X_train, y_train, cv=cv, scoring=scoring,
            n_jobs=1, error_score="raise", return_train_score=False
        )
        row = {"Sampler": sampler_name}
        for metric in scoring:
            vals = scores[f"test_{metric}"]
            row[metric + "_mean"] = np.mean(vals)
            row[metric + "_std"] = np.std(vals)
        cv_results.append(row)
    except Exception as e:
        print("Skipped due to error:", repr(e))

cv_table = pd.DataFrame(cv_results).sort_values("bal_acc_mean", ascending=False)
display(cv_table)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=cv_table, x="bal_acc_mean", y="Sampler")
plt.xlabel("Repeated-CV balanced accuracy (mean)")
plt.ylabel("")
plt.title(f"Sampler comparison - {MODEL_NAME}")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 9. Tune CKD-AHSS design choices on training data only

This grid tests different target ratios and safety thresholds. Each configuration is evaluated on the same repeated stratified folds. Keep the grid modest initially; increase it after confirming the code runs.

In [ ]:
ahss_configs = [
    {"sampling_strategy": ratio, "safety_min": safety, "hardness_focus": hardness,
     "n_clusters": clusters, "clean_majority": clean}
    for ratio in [0.35, 0.50, 0.67, 1.00]
    for safety in [0.15, 0.25, 0.35]
    for hardness in [0.25, 0.50, 0.75]
    for clusters in [2, 4]
    for clean in [False]
]

tune_cv = RepeatedStratifiedKFold(n_splits=4, n_repeats=2, random_state=RANDOM_STATE)
tuning_rows = []
for i, cfg in enumerate(ahss_configs, 1):
    sampler = CKDAHSSSampler(k_neighbors=5, random_state=RANDOM_STATE, **cfg)
    pipe = make_pipeline(sampler, MODEL_NAME, X_reference=X_train)
    scores = cross_validate(
        pipe, X_train, y_train, cv=tune_cv,
        scoring={"bal_acc": make_scorer(balanced_accuracy_score),
                 "recall": make_scorer(recall_score, pos_label=1, zero_division=0)},
        n_jobs=1, error_score="raise"
    )
    tuning_rows.append({
        **cfg,
        "bal_acc_mean": scores["test_bal_acc"].mean(),
        "bal_acc_std": scores["test_bal_acc"].std(),
        "recall_mean": scores["test_recall"].mean()
    })
    if i % 10 == 0:
        print(f"Completed {i}/{len(ahss_configs)}")

tuning_table = pd.DataFrame(tuning_rows).sort_values(
    ["bal_acc_mean", "recall_mean"], ascending=False
).reset_index(drop=True)
display(tuning_table.head(15))

best_row = tuning_table.iloc[0]
best_cfg = {
    "sampling_strategy": float(best_row["sampling_strategy"]),
    "safety_min": float(best_row["safety_min"]),
    "hardness_focus": float(best_row["hardness_focus"]),
    "n_clusters": int(best_row["n_clusters"]),
    "clean_majority": bool(best_row["clean_majority"]),
}
print("Best training-CV configuration:", best_cfg)

## 10. Train the selected model and evaluate once on the untouched test set

The sampler configuration is selected using training-side CV. The test set is now used for one final evaluation. This cell uses threshold 0.5; threshold tuning should be done with out-of-fold training predictions, not test predictions.

In [ ]:
final_sampler = CKDAHSSSampler(k_neighbors=5, random_state=RANDOM_STATE, **best_cfg)
final_pipe = make_pipeline(final_sampler, MODEL_NAME, X_reference=X_train)
final_pipe.fit(X_train, y_train)

test_prob = final_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.50).astype(int)

def report_metrics(y_true, y_pred, y_prob=None):
    out = {
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "CKD recall/sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Non-CKD specificity": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "CKD precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "CKD F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }
    if y_prob is not None:
        out["ROC-AUC"] = roc_auc_score(y_true, y_prob)
        out["PR-AUC (CKD positive)"] = average_precision_score(y_true, y_prob)
    return pd.Series(out)

display(report_metrics(y_test, test_pred, test_prob).to_frame("Test result"))
print(classification_report(y_test, test_pred, target_names=["Non-CKD", "CKD"], zero_division=0))

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Non-CKD", "Pred CKD"],
            yticklabels=["Actual Non-CKD", "Actual CKD"])
plt.title("CKD-AHSS test confusion matrix")
plt.tight_layout()
plt.show()

## 11. Training-side threshold optimization (optional)

This produces out-of-fold probabilities on the training partition, then selects the threshold maximizing balanced accuracy. The untouched test set is evaluated once using that selected threshold.

For strict model-selection studies, threshold selection and hyperparameter selection should be nested within the training process.

In [ ]:
oof_prob = np.zeros(len(X_train), dtype=float)
threshold_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, val_idx) in enumerate(threshold_cv.split(X_train, y_train), 1):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr = y_train.iloc[tr_idx]
    fold_pipe = make_pipeline(
        CKDAHSSSampler(k_neighbors=5, random_state=RANDOM_STATE, **best_cfg),
        MODEL_NAME, X_reference=Xtr
    )
    fold_pipe.fit(Xtr, ytr)
    oof_prob[val_idx] = fold_pipe.predict_proba(Xval)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
threshold_scores = [
    balanced_accuracy_score(y_train, (oof_prob >= t).astype(int))
    for t in thresholds
]
best_threshold = float(thresholds[int(np.argmax(threshold_scores))])
print("Selected training-OOF threshold:", best_threshold)
print("Training OOF balanced accuracy at selected threshold:", max(threshold_scores))

test_pred_tuned = (test_prob >= best_threshold).astype(int)
display(report_metrics(y_test, test_pred_tuned, test_prob).to_frame("Test result - OOF threshold"))

## 12. Ablation study

Ablation asks whether each proposed component contributes value. Compare the full sampler against variants with safety filtering, hardness weighting, clustering, and majority cleaning disabled. Do not claim that all components help unless the results support it.

In [ ]:
ablation_configs = {
    "Full CKD-AHSS": dict(safety_min=best_cfg["safety_min"], hardness_focus=best_cfg["hardness_focus"],
                          n_clusters=best_cfg["n_clusters"], clean_majority=False),
    "No safety filtering": dict(safety_min=0.0, hardness_focus=best_cfg["hardness_focus"],
                                n_clusters=best_cfg["n_clusters"], clean_majority=False),
    "No hardness weighting": dict(safety_min=best_cfg["safety_min"], hardness_focus=0.0,
                                  n_clusters=best_cfg["n_clusters"], clean_majority=False),
    "No cluster restriction": dict(safety_min=best_cfg["safety_min"], hardness_focus=best_cfg["hardness_focus"],
                                   n_clusters=1, clean_majority=False),
    "With majority cleaning": dict(safety_min=best_cfg["safety_min"], hardness_focus=best_cfg["hardness_focus"],
                                   n_clusters=best_cfg["n_clusters"], clean_majority=True)
}

ablation_rows = []
ablation_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
for name, cfg in ablation_configs.items():
    sampler = CKDAHSSSampler(
        sampling_strategy=best_cfg["sampling_strategy"],
        k_neighbors=5, random_state=RANDOM_STATE, **cfg
    )
    pipe = make_pipeline(sampler, MODEL_NAME, X_reference=X_train)
    scores = cross_validate(pipe, X_train, y_train, cv=ablation_cv,
                            scoring="balanced_accuracy", n_jobs=1, error_score="raise")
    ablation_rows.append({"Variant": name,
                          "Balanced accuracy mean": scores["test_score"].mean(),
                          "Balanced accuracy std": scores["test_score"].std()})
ablation_table = pd.DataFrame(ablation_rows).sort_values("Balanced accuracy mean", ascending=False)
display(ablation_table)

## 13. Save experiment outputs

These CSV files can be used to prepare tables for your paper. Keep the random seed, package versions, split strategy, and dataset version with your research records.

In [ ]:
import sklearn, imblearn
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

cv_table.to_csv("sampler_comparison_cv.csv", index=False)
tuning_table.to_csv("ckd_ahss_tuning_results.csv", index=False)
ablation_table.to_csv("ckd_ahss_ablation_results.csv", index=False)
report_metrics(y_test, test_pred, test_prob).to_csv("ckd_ahss_final_test_metrics_threshold_0.5.csv")
report_metrics(y_test, test_pred_tuned, test_prob).to_csv("ckd_ahss_final_test_metrics.csv")
print("Saved CSV result files.")

## 14. How to report the method responsibly

Suggested manuscript wording (edit after obtaining actual results):

> We developed CKD-AHSS, an experimental adaptive hybrid sampling approach that combines neighbourhood-based minority safety assessment, cluster-constrained mixed-type synthetic generation, hardness-aware anchor selection, optional conservative majority cleaning, and a configurable sampling ratio. All resampling operations were restricted to training folds. The method was compared with established resampling baselines using repeated stratified cross-validation and an untouched hold-out test set.

### Limitations to disclose
- Neighbourhood scores are sensitive to distance definitions and the small minority sample count.
- Clustering and interpolation can still create unrealistic clinical combinations.
- The proposed method is not guaranteed to outperform established methods.
- A single small hold-out test set is insufficient for strong clinical generalization claims.
- External validation on a larger, independent cohort is needed.

### Before publication
1. Verify every class label and categorical variable.
2. Inspect synthetic records for clinically impossible values.
3. Repeat the study with multiple random seeds and repeated/nested CV.
4. Report confidence intervals and paired statistical comparisons.
5. Include sampler ablations and computational cost.
6. Validate on an independent dataset if available.